# FAERS 耳毒性信号检测 · 主 notebook（清爽版）

**从上往下依次运行即可**——已保证"先定义、后使用"，重启内核后 Run All 不会报错。

分区：
1. **工具准备**：导入库、配置
2. **信号引擎**：算 ROR/PRR 的函数（正式分析全靠它）
3. **毒性定义层（唯一真源）**：最终口径 = `primary_cochlear`(10词) + `primary_vestibular`(3词)
4. **术语核对**：确认最终 10 个词在 FAERS 里真实存在
5. **正式分析**：只用最终定稿口径
6. **附录·历史探索**：当初怎么筛出最终口径（①–⑤ 对比、补词验证），仅作记录
7. **归档**：上一代 stage-0 代码 & 踩坑记录 & 下一步(1d)

**一条约定**：正式结果一律认 `FINAL_COCHLEAR` / `primary_cochlear`；`pt_set()` 取的是**探索草稿**，只在附录对比里用。

## 1. 工具准备
导入库、设置全局配置。

In [1]:
# requests 连网查 openFDA；math 算数；time 控制查询节奏；pandas 存表格
import requests, math, time
import pandas as pd

BASE = "https://api.fda.gov/drug/event.json"   # openFDA 药物不良事件接口
API_KEY = None    # 有 openFDA API key 就填进引号里提高配额；没有就保持 None
SLEEP = 0.3       # 每次查询后歇 0.3 秒，避免请求太密被限流(429)

print("工具准备好了")

工具准备好了


## 2. 信号引擎（先定义，后面才能用）

这几个函数负责"数报告数 → 算 ROR/PRR"。你不用看懂每一行，会调用就行。

**一个关键修法**：`_count` 里是自己手拼完整 URL，而**不用** `requests` 的 `params=`。
因为 openFDA 的 `+AND+` 一旦交给 `params=` 会被二次编码成 `%2BAND%2B`，导致**静默返回 0**（这是早期踩过的坑）。

In [2]:
def _count(search_query, retries=3):
    """给一个查询条件，返回匹配的报告数。带自动重试，应对偶发网络失败。"""
    url = f"{BASE}?search={search_query}&limit=1"     # 手拼 URL，让 +AND+ 原样发出
    if API_KEY:
        url += f"&api_key={API_KEY}"
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=30)          # 最多等 30 秒，别无限挂着
            time.sleep(SLEEP)
            if r.status_code == 404:                    # 没匹配到任何记录，当作 0
                return 0
            if r.status_code == 429:                    # 被限流，多等一会儿再试
                time.sleep(2 * (attempt + 1))
                continue
            r.raise_for_status()
            return r.json()["meta"]["results"]["total"]
        except requests.exceptions.RequestException:
            if attempt == retries - 1:                  # 试满 3 次还不行，才真报错
                raise
            time.sleep(2 * (attempt + 1))               # 等 2 秒、4 秒… 再试


def _total_db():
    """数据库总报告数 N（不带任何查询条件）。"""
    url = f"{BASE}?limit=1"
    if API_KEY:
        url += f"&api_key={API_KEY}"
    r = requests.get(url, timeout=30); time.sleep(SLEEP); r.raise_for_status()
    return r.json()["meta"]["results"]["total"]


def reaction_or_clause(pt_list):
    """把一组 PT 拼成 openFDA 的 OR 条件。
    .exact 保证整词匹配；词里的空格换成 +，否则 'SUDDEN HEARING LOSS' 会被拆成三个词分别计数。"""
    terms = "+".join(f'"{pt.replace(" ", "+")}"' for pt in pt_list)
    return f"patient.reaction.reactionmeddrapt.exact:({terms})"


def drug_clause(name):
    """药物条件。用规范化的 generic_name.exact（已验证顺铂能复现已知信号）。"""
    return f'patient.drug.openfda.generic_name.exact:"{name}"'


def ror_prr(drug, pt_list, N=None):
    """对 (药 × 一组事件PT) 算 ROR、PRR 及 95%CI。
    信号判定沿用：95%CI 下限 > 1 且 a >= 3。"""
    d = drug_clause(drug)
    e = reaction_or_clause(pt_list)

    a = _count(f"{d}+AND+{e}")          # a：药 且 事件
    drug_total = _count(d)              # a+b：所有该药报告
    event_total = _count(e)             # a+c：所有该事件报告
    if N is None:
        N = _total_db()                 # N：数据库总报告数

    b = drug_total - a
    c = event_total - a
    dd = N - drug_total - event_total + a

    # 样本太少或某格为空，信号不成立
    if a < 3 or b <= 0 or c <= 0 or dd <= 0:
        return {"药": drug, "a": a, "b": b, "c": c, "d": dd, "说明": "样本不足或空单元"}

    ror = (a * dd) / (b * c)
    se = math.sqrt(1/a + 1/b + 1/c + 1/dd)
    lo = math.exp(math.log(ror) - 1.96 * se)
    hi = math.exp(math.log(ror) + 1.96 * se)
    prr = (a / (a + b)) / (c / (c + dd))

    return {"药": drug, "PT数": len(pt_list), "a": a,
            "ROR": round(ror, 2), "95%CI": (round(lo, 2), round(hi, 2)),
            "PRR": round(prr, 2), "信号": (lo > 1 and a >= 3)}

print("引擎装好了")

引擎装好了


## 3. 毒性定义层（唯一真源）

整个项目只在这里定义"哪些 PT 算耳毒性"。**换毒性 = 在这里加一个词条，引擎不用改。**

- **定稿（正式分析用）**：`primary_cochlear`(10词) + `primary_vestibular`(3词)
- **草稿（只在附录①–⑤对比里用）**：`cochlear_core` 等，含 TINNITUS，是当初探索留下的宽口径

为防止自己用错，另设 `FINAL_COCHLEAR` / `FINAL_VESTIBULAR` 两个"带 FINAL 的"变量，一眼认出是定稿。

In [3]:
OTOTOXICITY = {
    # ============ 最终定稿：正式分析只用下面这两个 ============
    # 耳蜗/听觉 主口径（10 个特异 PT）：经官方 SMQ 20000171 核对、数据验证"补全而不稀释"
    "primary_cochlear": [
        "DEAFNESS", "DEAFNESS NEUROSENSORY", "NEUROSENSORY HYPOACUSIS",
        "DEAFNESS BILATERAL", "DEAFNESS UNILATERAL", "DEAFNESS PERMANENT",
        "DEAFNESS TRANSITORY", "SUDDEN HEARING LOSS", "HYPOACUSIS", "OTOTOXICITY",
    ],
    # 前庭/平衡 副维度：单独报告，不并入主口径（顺铂"耳蜗强、前庭无"本身就是结果）
    "primary_vestibular": [
        "VERTIGO", "VERTIGO POSITIONAL", "BALANCE DISORDER",
    ],

    # ============ 探索草稿：仅供附录①–⑤对比，非最终，别用于正式分析 ============
    "cochlear_core": [   # 当初探索用：含 TINNITUS 的 9 词（正是它让信号被稀释，才淘汰了 tinnitus）
        "DEAFNESS", "DEAFNESS NEUROSENSORY", "DEAFNESS BILATERAL",
        "DEAFNESS UNILATERAL", "DEAFNESS PERMANENT", "SUDDEN HEARING LOSS",
        "HYPOACUSIS", "TINNITUS", "OTOTOXICITY",
    ],
    "vestibular_core": ["VERTIGO", "VERTIGO POSITIONAL", "BALANCE DISORDER"],
    "cochlear_extended": ["HYPERACUSIS", "AUDITORY DISTURBANCE"],   # 仅敏感性分析用
    "vestibular_extended": ["DIZZINESS", "LABYRINTHITIS"],          # 仅敏感性分析用（不特异）
}

# —— 防呆变量：正式分析一律用这两个，名字带 FINAL，一眼知道是定稿 ——
FINAL_COCHLEAR   = OTOTOXICITY["primary_cochlear"]
FINAL_VESTIBULAR = OTOTOXICITY["primary_vestibular"]

# —— 定义来源（写论文可引用；比版本号更稳，因为 SMQ 代码不随版本变）——
MEDDRA_SOURCE = ("耳毒性 PT 依 SMQ 20000171 (Hearing impairment)，"
                 "取感觉神经性药物耳毒子集，剔除 presbyacusis 等混杂/结构/检查类 PT；"
                 "参照 Liu et al. 2025, Front Pharmacol, DOI 10.3389/fphar.2025.1532461, Table S1")


def pt_set(tox, include_extended=False, dimension="all"):
    """取【探索草稿】词单——只在附录①–⑤对比里用。正式分析请用 FINAL_COCHLEAR / FINAL_VESTIBULAR。"""
    pts = []
    if dimension in ("all", "cochlear"):
        pts += tox["cochlear_core"] + (tox["cochlear_extended"] if include_extended else [])
    if dimension in ("all", "vestibular"):
        pts += tox["vestibular_core"] + (tox["vestibular_extended"] if include_extended else [])
    return pts


print("定义层就绪 | 定稿耳蜗", len(FINAL_COCHLEAR), "词 | 定稿前庭", len(FINAL_VESTIBULAR), "词")

定义层就绪 | 定稿耳蜗 10 词 | 定稿前庭 3 词


## 4. 术语核对
确认定稿的 10 个词在 FAERS 里都真实存在、拼写无误（报告数都不为 0）。

In [4]:
print("逐个检查定稿 10 个词在 FAERS 里存不存在：\n")
for pt in FINAL_COCHLEAR:
    q = f'patient.reaction.reactionmeddrapt.exact:"{pt.replace(" ", "+")}"'
    n = _count(q)
    flag = "查不到——核对拼写/是否为PT" if n == 0 else "OK"
    print(f"{pt:26s} 报告数={n:>8}   {flag}")

逐个检查定稿 10 个词在 FAERS 里存不存在：

DEAFNESS                   报告数=   25391   OK
DEAFNESS NEUROSENSORY      报告数=    3166   OK
NEUROSENSORY HYPOACUSIS    报告数=     117   OK
DEAFNESS BILATERAL         报告数=    1640   OK
DEAFNESS UNILATERAL        报告数=    5346   OK
DEAFNESS PERMANENT         报告数=     195   OK
DEAFNESS TRANSITORY        报告数=     838   OK
SUDDEN HEARING LOSS        报告数=    1893   OK
HYPOACUSIS                 报告数=   41604   OK
OTOTOXICITY                报告数=    1794   OK


## 5. 正式分析（只用定稿口径）
正式结果一律用 `FINAL_COCHLEAR` / `FINAL_VESTIBULAR`。

In [5]:
# 顺铂作为阳性对照锚点：耳蜗应强阳性，前庭应弱/不成信号
print("顺铂 · 耳蜗主口径(定稿10词):", ror_prr("CISPLATIN", FINAL_COCHLEAR))
print("顺铂 · 前庭副维度(单独看)  :", ror_prr("CISPLATIN", FINAL_VESTIBULAR))

顺铂 · 耳蜗主口径(定稿10词): {'药': 'CISPLATIN', 'PT数': 10, 'a': 1070, 'ROR': 3.67, '95%CI': (3.46, 3.9), 'PRR': 3.64, '信号': True}
顺铂 · 前庭副维度(单独看)  : {'药': 'CISPLATIN', 'PT数': 3, 'a': 178, 'ROR': 0.33, '95%CI': (0.29, 0.39), 'PRR': 0.34, '信号': False}


## 6. 附录 · 历史探索（如何筛出最终口径）

下面两格是"怎么得出定稿"的**记录**，结论已经写进第 3 节的 `primary_cochlear`，这里只为留痕、写论文时可引用。

**注意**：②④ 用的是 `pt_set()` 取的**草稿**（含 tinnitus / 前庭），不是定稿——标签里已注明。

In [6]:
# ①–⑤ 五个口径对比：说明为什么最终选"特异耳蜗核心"、且把前庭单列
print("① 单个 DEAFNESS           :", ror_prr("CISPLATIN", ["DEAFNESS"]))
print("② 耳蜗草稿(9词,含tinnitus) :", ror_prr("CISPLATIN", pt_set(OTOTOXICITY, dimension="cochlear")))
print("③ 前庭(3词)               :", ror_prr("CISPLATIN", FINAL_VESTIBULAR))
print("④ 草稿全部(耳蜗+前庭)       :", ror_prr("CISPLATIN", pt_set(OTOTOXICITY)))
print("⑤ 定稿耳蜗主口径(10词)      :", ror_prr("CISPLATIN", FINAL_COCHLEAR))
# 结论：④加入前庭/不特异词后 ROR 被稀释(→1.54)；③前庭本就不成信号(阴性对照)；
#       ⑤定稿既补全病例又不稀释(ROR≈3.67) → 选⑤为主口径、前庭单列。

① 单个 DEAFNESS           : {'药': 'CISPLATIN', 'PT数': 1, 'a': 404, 'ROR': 4.31, '95%CI': (3.91, 4.76), 'PRR': 4.3, '信号': True}
② 耳蜗草稿(9词,含tinnitus) : {'药': 'CISPLATIN', 'PT数': 9, 'a': 1290, 'ROR': 3.01, '95%CI': (2.84, 3.18), 'PRR': 2.97, '信号': True}
③ 前庭(3词)               : {'药': 'CISPLATIN', 'PT数': 3, 'a': 178, 'ROR': 0.33, '95%CI': (0.29, 0.39), 'PRR': 0.34, '信号': False}
④ 草稿全部(耳蜗+前庭)       : {'药': 'CISPLATIN', 'PT数': 12, 'a': 1432, 'ROR': 1.54, '95%CI': (1.46, 1.62), 'PRR': 1.53, '信号': True}
⑤ 定稿耳蜗主口径(10词)      : {'药': 'CISPLATIN', 'PT数': 10, 'a': 1070, 'ROR': 3.67, '95%CI': (3.46, 3.9), 'PRR': 3.64, '信号': True}


In [7]:
# 补词验证：定稿(10词)相比旧8词，多捞回病例但 ROR 不掉 → "补全而不稀释"
old8 = ["DEAFNESS", "DEAFNESS NEUROSENSORY", "DEAFNESS BILATERAL", "DEAFNESS UNILATERAL",
        "DEAFNESS PERMANENT", "SUDDEN HEARING LOSS", "HYPOACUSIS", "OTOTOXICITY"]
print("旧 8 词      :", ror_prr("CISPLATIN", old8))
print("新10词(定稿) :", ror_prr("CISPLATIN", FINAL_COCHLEAR))
# 实测：a 从 1052 → 1070（多捞回病例），ROR 3.65 → 3.67（几乎不变）。

旧 8 词      : {'药': 'CISPLATIN', 'PT数': 8, 'a': 1052, 'ROR': 3.65, '95%CI': (3.43, 3.88), 'PRR': 3.61, '信号': True}
新10词(定稿) : {'药': 'CISPLATIN', 'PT数': 10, 'a': 1070, 'ROR': 3.67, '95%CI': (3.46, 3.9), 'PRR': 3.64, '信号': True}


## 7. 归档 · 上一代代码 & 踩坑记录 & 下一步(1d)

**踩坑记录（重要，别再犯）**：早期用 `requests` 的 `params=` 传 `+AND+`，会被二次编码成 `%2BAND%2B`，openFDA **静默返回 0**、不报错。修法就是本 notebook `_count` 的做法：自己拼完整 URL 字符串、不用 `params=`。**任何新查询模式，都先用"顺铂×耳聋 ROR≈4.3"这个已知答案验一遍再信。**

**上一代 stage-0 代码**（手算 2×2、`get_total`、`analyze`、单事件的 7×3 矩阵）仍在你原始的 `Untitled.ipynb` 里，作为学习历史保留，本清爽版不再收录。

**下一步 1d**：正式批量矩阵要用**本 notebook 的新引擎 `ror_prr` + 定稿口径**重搭，别用老 `analyze`（它没 `.exact`、也没用 SMQ 定义）。下面给一个骨架示例（先注释掉，1d 时再启用/扩展）。

In [8]:
# ===== 1d 预备：用新引擎 + 定稿口径的批量矩阵骨架（先注释，1d 再用）=====
# drugs = ["cisplatin","gentamicin","furosemide","vancomycin","amikacin",  # 阳性
#          "metformin","atorvastatin"]                                      # 阴性对照
# N = _total_db()   # 总数只取一次，循环里复用，省查询更稳
# rows = []
# for drug in drugs:
#     rows.append(ror_prr(drug.upper(), FINAL_COCHLEAR, N=N))   # 耳蜗主口径
#     print(drug, "完成")
# df = pd.DataFrame(rows)
# print(df)
print("骨架已就绪（当前为注释状态），1d 时启用。")

骨架已就绪（当前为注释状态），1d 时启用。
